# Bias and variance of a bag, measured

The lesson's formula says that averaging $B$ models leaves a variance of
$\rho\sigma^2 + (1 - \rho)\sigma^2 / B$ and says nothing about bias. Here you measure both.
The data comes from a known curve plus noise, so you can draw as many training sets as you
like, fit a model to each, and see how its predictions at fixed test points spread around
the truth. You will write the bias and variance estimates, a bag of regression trees, and
the experiment that compares one deep tree with a bag of them.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and
the `raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says what is
wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

x_test = np.linspace(-3, 3, 41)          # the fixed points where predictions are compared

def truth(x):
    """The curve every training set is drawn from."""
    return np.sin(1.5 * x) + 0.3 * x

def draw_training_set(rng, n=80, noise=0.6):
    """n rows from the curve with Gaussian noise: X has shape (n, 1)."""
    x = rng.uniform(-3, 3, n)
    return x.reshape(-1, 1), truth(x) + noise * rng.normal(size=n)

X_demo, y_demo = draw_training_set(np.random.default_rng(0))
print(f"one training set: X {X_demo.shape}, y {y_demo.shape}, noise variance 0.36")

## Part 1 (coding task): Bias and variance from repeated fits

Write `bias_variance(preds, target)`. Row `s` of `preds` holds a model's predictions at
the test points after training on training set `s`, so `preds` has shape (sets, points),
and `target` holds the true value at each point. At each point, the squared bias is the
square of (average prediction over the sets minus the truth), and the variance is the
spread of the predictions over the sets (`np.var`, dividing by the number of sets). Return
the pair `(bias2, variance)`, each averaged over the points.

In [ ]:
def bias_variance(preds, target):
    """Squared bias and variance at each test point, averaged over the points."""
    preds = np.asarray(preds, dtype=float)
    target = np.asarray(target, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
b2, v = bias_variance([[1.0, 2.0, 0.0], [3.0, 2.0, 4.0]], [1.0, 1.0, 1.0])
assert np.isclose(b2, 1.0) and np.isclose(v, 5 / 3), f"got ({b2}, {v}), expected (1, 5/3)"
b2, v = bias_variance([[0.0, 0.0, 0.0, 0.0], [2.0, 2.0, 2.0, 2.0]], [1.0, 1.0, 1.0, 1.0])
assert np.isclose(v, 1.0), f"variance {v}: at every point the two sets give 0 and 2, so the variance is 1. Take it over the sets (axis 0), not over the points"
b2, v = bias_variance([[0.0, 4.0], [0.0, 4.0]], [0.0, 4.0])
assert np.isclose(v, 0.0) and np.isclose(b2, 0.0), f"got ({b2}, {v}): both sets predict the truth exactly, so bias and variance are 0. The spread across points is not variance"
b2, v = bias_variance([[2.0, 0.0], [2.0, 0.0]], [0.0, 2.0])
assert np.isclose(b2, 4.0), f"squared bias {b2}: the errors +2 and -2 must be squared point by point before averaging, giving 4"
print("Part 1 passed")

## Part 2 (coding task): A bag of regression trees

Write `bagged_predict(X, y, x_new, B, rng)`. Fit `B` trees, each a
`DecisionTreeRegressor(random_state=0)` grown to full depth on its own bootstrap sample of
the rows, and return the average of their predictions at the points `x_new` (a 1-D array).
Draw each tree's sample with `rng.integers(0, n, size=n)`, one tree after another, so that
your result can be compared with the expected one.

In [ ]:
def bagged_predict(X, y, x_new, B, rng):
    """Average prediction at x_new of B deep trees, each fit on a bootstrap sample."""
    n = len(y)
    x_new = np.asarray(x_new, dtype=float).reshape(-1, 1)
    # TODO
    raise NotImplementedError

In [ ]:
Xs = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
ys = np.array([0.0, 2.0, 1.0, 3.0, 5.0, 4.0])
x_new = [0.4, 1.4, 2.6, 4.9]
full = DecisionTreeRegressor(random_state=0).fit(Xs, ys).predict(np.reshape(x_new, (-1, 1)))
got = bagged_predict(Xs, ys, x_new, 4, np.random.default_rng(1))
assert np.shape(got) == (4,), f"expected one prediction per point, shape (4,), got {np.shape(got)}"
assert not np.allclose(got, full), "the bag predicts exactly what one tree on all the rows predicts: draw each sample with replacement"
assert np.allclose(got, [1.0, 1.25, 2.5, 4.25]), f"got {got}, expected [1.0, 1.25, 2.5, 4.25]: draw a new sample for every tree and average the B predictions"
print("Part 2 passed")

## Part 3 (coding task): One tree against a bag

Write `simulate(S, B, rng)`. Repeat `S` times: draw a new training set with
`draw_training_set(rng)`, fit one deep `DecisionTreeRegressor(random_state=0)` to it and
predict at `x_test`, and get the bag's predictions at `x_test` with
`bagged_predict(..., B, rng)`. Return a dictionary
`{"tree": (bias2, variance), "bag": (bias2, variance)}` measured against `truth(x_test)`.

Then write `implied_correlation(v_tree, v_bag, B)`: if every tree in the bag had the deep
tree's variance `v_tree`, which correlation $\rho$ between two trees would make the
average of `B` of them have variance `v_bag`?

In [ ]:
def simulate(S, B, rng):
    """Bias and variance of one deep tree and of a bag of B trees, over S training sets."""
    # TODO
    raise NotImplementedError


def implied_correlation(v_tree, v_bag, B):
    # TODO
    raise NotImplementedError

In [ ]:
assert np.isclose(implied_correlation(1.0, 0.3, 10), 2 / 9), "implied_correlation(1, 0.3, 10) should be 2/9: solve v_bag = rho v + (1 - rho) v / B for rho"
assert np.isclose(implied_correlation(2.0, 2.0, 5), 1.0), "a bag no better than one tree means rho = 1"

res = simulate(30, 20, np.random.default_rng(4))
(tb, tv), (bb, bv) = res["tree"], res["bag"]
print(f"one deep tree   bias^2 {tb:.4f}   variance {tv:.4f}")
print(f"bag of 20       bias^2 {bb:.4f}   variance {bv:.4f}")
assert tv > 0.2, f"tree variance {tv:.4f}: draw a new training set in every repeat"
rho = implied_correlation(tv, bv, 20)
print(f"implied correlation between the bag's trees {rho:.2f}")
assert 0.3 * tv < bv < 0.7 * tv, f"bag variance {bv:.4f} against the tree's {tv:.4f}: expected the bag to cut it to roughly half"
assert bb < 0.02 and tb < 0.03, "both bias^2 values should be small: a deep tree has little bias, and averaging keeps it small"
assert 0.3 < rho < 0.7, f"implied correlation {rho:.2f}, expected between 0.3 and 0.7"
print("Part 3 passed")

## Solution

### Part 1: Bias and variance from repeated fits

In [ ]:
def bias_variance(preds, target):
    """Squared bias and variance at each test point, averaged over the points."""
    preds = np.asarray(preds, dtype=float)
    target = np.asarray(target, dtype=float)
    mean_pred = preds.mean(axis=0)                    # average over the training sets
    bias2 = np.mean((mean_pred - target) ** 2)
    variance = np.mean(preds.var(axis=0))
    return bias2, variance

**Why it works.** The expectation in the bias and the variance is over training sets, so both are
taken down axis 0, one test point at a time, and only then averaged over the points. The
checks catch the three usual slips: the variance taken along the points, the variance of
the flattened array (which counts the curve's own rise and fall as variance), and the
errors averaged before squaring, which lets errors of opposite sign cancel.

### Part 2: A bag of regression trees

In [ ]:
def bagged_predict(X, y, x_new, B, rng):
    """Average prediction at x_new of B deep trees, each fit on a bootstrap sample."""
    n = len(y)
    x_new = np.asarray(x_new, dtype=float).reshape(-1, 1)
    total = np.zeros(len(x_new))
    for _ in range(B):
        rows = rng.integers(0, n, size=n)             # n draws with replacement
        tree = DecisionTreeRegressor(random_state=0).fit(X[rows], y[rows])
        total += tree.predict(x_new)
    return total / B

**Why it works.** A deep tree reproduces its training rows, so a tree fit on all six rows, or on any
reordering of them, predicts the same thing. A sample drawn without replacement is such a
reordering, and a bag of those trees is one tree. Drawing with replacement leaves out about
a third of the rows and repeats others, so each tree is different, and the average of the
`B` trees smooths their disagreements.

### Part 3: One tree against a bag

In [ ]:
def simulate(S, B, rng):
    """Bias and variance of one deep tree and of a bag of B trees, over S training sets."""
    tree_preds, bag_preds = [], []
    for _ in range(S):
        X, y = draw_training_set(rng)                 # a new training set every repeat
        tree = DecisionTreeRegressor(random_state=0).fit(X, y)
        tree_preds.append(tree.predict(x_test.reshape(-1, 1)))
        bag_preds.append(bagged_predict(X, y, x_test, B, rng))
    target = truth(x_test)
    return {"tree": bias_variance(tree_preds, target), "bag": bias_variance(bag_preds, target)}


def implied_correlation(v_tree, v_bag, B):
    # v_bag = rho * v_tree + (1 - rho) * v_tree / B, solved for rho
    return (v_bag / v_tree - 1 / B) / (1 - 1 / B)

**Why it works.** Each repeat stands for one draw of the training data, so the spread of the
predictions across repeats is the variance a model would show if you collected the data
again. A deep tree on 80 noisy rows has almost no bias and a variance near the noise
level. The bag of 20 halves the variance and does not add bias: all
of its gain is variance. It does not get anywhere near $1/20$ of the tree's variance,
because its trees all saw much the same rows and correlate at about 0.5, and
$\rho\sigma^2$ is the floor that no number of trees removes. The implied correlation is a
rough reading, since a tree on a bootstrap sample is a little noisier than a tree on all
the rows. The measured squared bias is mostly noise: the average prediction is taken over
only 30 training sets, so its square carries about variance over $S$, near 0.35 / 30 for the
tree and 0.18 / 30 for the bag. That is as large as the measured values, and it is why the
bag's comes out lower than the tree's.